In [ ]:
!pip install --upgrade "google-cloud-aiplatform[agent_engines,adk]>=1.112.0" cloudpickle pydantic

# Develop an agent

In [ ]:
import os
from dotenv import load_dotenv
import agentplatform
from agentplatform.frameworks import AdkApp
from google.adk.agents import Agent

# 루트 .env 파일 로드
load_dotenv(override=True)
GEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemini-3.8-flash")

def get_exchange_rate(
    currency_from: str = "USD",
    currency_to: str = "EUR",
    currency_date: str = "latest",
):
    """Retrieves the exchange rate between two currencies on a specified date."""
    import requests

    response = requests.get(
        f"https://api.frankfurter.app/{currency_date}",
        params={"from": currency_from, "to": currency_to},
    )
    return response.json()

# Agent 및 App 정의
agent = Agent(
    model=GEMINI_MODEL,
    name="currency_exchange_agent",
    tools=[get_exchange_rate],
)

app = AdkApp(agent=agent)

# Deploy an agent to Agent Runtime

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

PROJECT_ID = os.getenv("GOOGLE_CLOUD_PROJECT") or input("Enter PROJECT_ID: ").strip()
LOCATION = os.getenv("GCP_RESOURCES_LOCATION", "us-central1")
STAGING_BUCKET = os.getenv("STAGING_BUCKET_URI") or input("Enter STAGING_BUCKET_URI (e.g. gs://...): ").strip()
GEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemini-3.8-flash")
SERVICE_ACCOUNT = os.getenv("SERVICE_ACCOUNT", f"sa-agent-platform@{PROJECT_ID}.iam.gserviceaccount.com")

print(f"Project: {PROJECT_ID}, Location: {LOCATION}, Bucket: {STAGING_BUCKET}, SA: {SERVICE_ACCOUNT}")

In [ ]:
import agentplatform

client = agentplatform.Client(
    project=PROJECT_ID,
    location=LOCATION,  
)

In [ ]:
remote_agent = client.runtimes.create(
    agent=app,
    config={
        "display_name": "currency_exchange_agent",
        "service_account": SERVICE_ACCOUNT,
        "requirements": [
            "google-cloud-aiplatform[agent_engines,adk]",
            "cloudpickle",
            "pydantic",
            "requests",
        ],
        "staging_bucket": STAGING_BUCKET,
        "env_vars": {
            "GOOGLE_CLOUD_LOCATION": "global",
            "GOOGLE_GENAI_USE_VERTEXAI": "TRUE",
            "GEMINI_MODEL": GEMINI_MODEL,
        },
    },
)

# Test the Agent on Agent Runtime

In [ ]:
async for event in remote_agent.async_stream_query(
    user_id="USER_ID",
    message="What is the exchange rate from US dollars to SEK today?",
):
    print(event)

In [ ]:
# 실습 완료 후 배포된 Reasoning Engine 삭제 (필요 시 주석 해제 후 실행)
# remote_agent.delete(force=True)